# Kaggle MoneyPrinterTurbo + IndicF5

This notebook starts two separate Python services: IndicF5 API (`8765`) and MoneyPrinterTurbo Streamlit (`8501`). It also exposes a direct Manual Timeline JSON generation function. Run cells in order. GPU and Internet must be enabled in Kaggle settings.

**Before first run:** add Kaggle Secrets `HF_TOKEN` (after accepting the `ai4bharat/IndicF5` model access conditions) and, after creating your private repo, `GITHUB_TOKEN`. Set `INDICF5_PRIVATE_REPO` below to `owner/repo`. Set a reference WAV and its exact transcript before running the Malayalam smoke test. Do not paste tokens into notebook cells.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, time, base64, shutil, textwrap
import requests

# ---- Project settings (safe to edit; never put tokens here) ----
MPT_REPO = "https://github.com/rjosephm/MoneyPrinterTurbo.git"
MPT_BRANCH = "feature/kaggle-indicf5-pipeline"
INDICF5_PRIVATE_REPO = ""  # After creating it, set e.g. "your-github-name/IndicF5"
INDICF5_BRANCH = "kaggle"
MODEL_DATASET_DIR = Path("/kaggle/input/indicf5-model")  # Optional private Kaggle Dataset containing a HF model snapshot
INDICF5_REFERENCE_AUDIO = ""  # e.g. /kaggle/input/my-voice/reference.wav
INDICF5_REFERENCE_TEXT = ""   # Exact transcript spoken in that WAV
ENABLE_PUBLIC_TUNNEL = False  # Set True only if you intentionally want a public Cloudflare URL for Streamlit

WORK = Path('/kaggle/working')
MPT_ROOT = WORK / 'MoneyPrinterTurbo'
PRIVATE_INDICF5_ROOT = WORK / 'IndicF5Private'
OUT_DIR = WORK / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE = WORK / 'hf-cache'
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE / 'hub')
os.environ['INDICF5_API_URL'] = 'http://127.0.0.1:8765'
os.environ['INDICF5_REFERENCE_AUDIO'] = INDICF5_REFERENCE_AUDIO
os.environ['INDICF5_REFERENCE_TEXT'] = INDICF5_REFERENCE_TEXT

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    def get_secret(name):
        try: return _secrets.get_secret(name)
        except Exception: return ''
except Exception:
    def get_secret(name): return ''

HF_TOKEN = get_secret('HF_TOKEN')
GITHUB_TOKEN = get_secret('GITHUB_TOKEN')
PEXELS_API_KEY = get_secret('PEXELS_API_KEY')
if PEXELS_API_KEY:
    os.environ['PEXELS_API_KEY'] = PEXELS_API_KEY
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN
else:
    print('WARNING: Kaggle Secret HF_TOKEN is missing. Gated Hugging Face model download may fail.')

print('Workspace:', WORK)
print('GPU check:')
subprocess.run([sys.executable, '-c', "import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No CUDA GPU')"], check=False)


In [ ]:
# Clone the selected MoneyPrinterTurbo feature branch.
if MPT_ROOT.exists():
    shutil.rmtree(MPT_ROOT)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', MPT_BRANCH, MPT_REPO, str(MPT_ROOT)], check=True)

# Optional private IndicF5 source. GitHub token is passed through Git's temporary environment config,
# not embedded in the clone URL and not printed to notebook output.
if INDICF5_PRIVATE_REPO.strip():
    if not GITHUB_TOKEN:
        raise RuntimeError('Set Kaggle Secret GITHUB_TOKEN with read-only access to your private IndicF5 repo.')
    if PRIVATE_INDICF5_ROOT.exists(): shutil.rmtree(PRIVATE_INDICF5_ROOT)
    auth = base64.b64encode(('x-access-token:' + GITHUB_TOKEN).encode()).decode()
    git_env = os.environ.copy()
    git_env['GIT_CONFIG_COUNT'] = '1'
    git_env['GIT_CONFIG_KEY_0'] = 'http.extraheader'
    git_env['GIT_CONFIG_VALUE_0'] = 'AUTHORIZATION: basic ' + auth
    private_url = 'https://github.com/' + INDICF5_PRIVATE_REPO.strip().removesuffix('.git') + '.git'
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', INDICF5_BRANCH, private_url, str(PRIVATE_INDICF5_ROOT)], env=git_env, check=True, capture_output=True, text=True)
    print('Private IndicF5 source cloned:', INDICF5_PRIVATE_REPO, 'branch', INDICF5_BRANCH)
else:
    print('Private IndicF5 repo not configured yet. Bootstrap uses the Kaggle API adapter in the MoneyPrinterTurbo branch; fill INDICF5_PRIVATE_REPO after creating your private repo.')

# Install the project's pinned dependencies and the known-compatible IndicF5 inference stack.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(MPT_ROOT / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'transformers==4.49.0', 'soundfile', 'safetensors', 'huggingface_hub'], check=True)

# Prefer a model snapshot attached as a Kaggle Dataset if one exists; otherwise use Hugging Face.
local_model = None
if MODEL_DATASET_DIR.exists():
    for cfg in MODEL_DATASET_DIR.rglob('config.json'):
        if list(cfg.parent.glob('*.safetensors')):
            local_model = cfg.parent
            break
if local_model:
    os.environ['INDICF5_MODEL_ID'] = str(local_model)
    print('Using model files from Kaggle Dataset:', local_model)
else:
    os.environ['INDICF5_MODEL_ID'] = 'ai4bharat/IndicF5'
    print('Using Hugging Face model id: ai4bharat/IndicF5')

# Prefer API source from the private repo if it includes indicf5_api.py; otherwise use the committed adapter.
fallback_api = MPT_ROOT / 'kaggle' / 'indicf5_api.py'
private_api_candidates = [PRIVATE_INDICF5_ROOT / 'indicf5_api.py', PRIVATE_INDICF5_ROOT / 'api' / 'indicf5_api.py']
INDICF5_API_SCRIPT = next((p for p in private_api_candidates if p.is_file()), fallback_api)
print('IndicF5 API script:', INDICF5_API_SCRIPT)
if not (MPT_ROOT / 'kaggle' / 'mpt_kaggle.py').is_file(): raise FileNotFoundError('Kaggle MPT launcher is missing from the selected branch.')


In [ ]:
# Start IndicF5 in its own Python process, then wait for the health endpoint.
API_LOG = open(WORK / 'indicf5_api.log', 'w', encoding='utf-8')
api_env = os.environ.copy()
api_env['INDICF5_HOST'] = '127.0.0.1'
api_env['INDICF5_PORT'] = '8765'
INDICF5_PROCESS = subprocess.Popen([sys.executable, str(INDICF5_API_SCRIPT)], cwd=str(INDICF5_API_SCRIPT.parent), env=api_env, stdout=API_LOG, stderr=subprocess.STDOUT)

health = None
deadline = time.time() + 900  # model download/loading can take several minutes
while time.time() < deadline:
    if INDICF5_PROCESS.poll() is not None:
        break
    try:
        response = requests.get('http://127.0.0.1:8765/health', timeout=3)
        if response.ok:
            health = response.json()
            if health.get('ok'): break
    except requests.RequestException:
        pass
    time.sleep(5)
print('IndicF5 health:', health)
if not health or not health.get('ok'):
    API_LOG.flush()
    print((WORK / 'indicf5_api.log').read_text(encoding='utf-8', errors='replace')[-12000:])
    raise RuntimeError('IndicF5 did not become healthy. Check HF_TOKEN/model access, dependencies, and the API log above.')
if not health.get('cuda_available'):
    raise RuntimeError('IndicF5 loaded without CUDA. Check Kaggle Settings → Accelerator → GPU before rendering.')

# Mandatory Malayalam smoke test. The reference WAV and exact transcript must be configured above.
if not INDICF5_REFERENCE_AUDIO or not Path(INDICF5_REFERENCE_AUDIO).is_file() or not INDICF5_REFERENCE_TEXT.strip():
    raise RuntimeError('Before continuing, set INDICF5_REFERENCE_AUDIO to an accessible WAV file and INDICF5_REFERENCE_TEXT to its exact transcript, then rerun this cell.')
test_response = requests.post('http://127.0.0.1:8765/generate', json={
    'text': 'നമസ്കാരം. ഇത് ഇൻഡിക് എഫ് ഫൈവ് ശബ്ദ പരിശോധനയാണ്.',
    'reference_audio': INDICF5_REFERENCE_AUDIO,
    'reference_text': INDICF5_REFERENCE_TEXT,
}, timeout=(10, 900))
test_response.raise_for_status()
if not test_response.content.startswith(b'RIFF') or test_response.content[8:12] != b'WAVE':
    raise RuntimeError('IndicF5 smoke test did not return a WAV file.')
SMOKE_WAV = WORK / 'indicf5_smoke_test.wav'
SMOKE_WAV.write_bytes(test_response.content)
if SMOKE_WAV.stat().st_size < 1000: raise RuntimeError('IndicF5 smoke-test WAV is unexpectedly small.')
print('PASS: IndicF5 health and Malayalam WAV test passed:', SMOKE_WAV, 'bytes:', SMOKE_WAV.stat().st_size)


In [ ]:
# Save the default IndicF5 profile so it appears in the Manual Timeline voice dropdown.
profile_dir = MPT_ROOT / 'storage' / 'indicf5_profiles'
profile_dir.mkdir(parents=True, exist_ok=True)
(profile_dir / 'kaggle-default.json').write_text(json.dumps({
    'id': 'kaggle-default', 'name': 'Kaggle IndicF5',
    'reference_audio': str(Path(INDICF5_REFERENCE_AUDIO).resolve()),
    'reference_text': INDICF5_REFERENCE_TEXT,
    'api_url': 'http://127.0.0.1:8765'
}, ensure_ascii=False, indent=2), encoding='utf-8')

# Start MoneyPrinterTurbo in a separate Python process. The IndicF5 process remains running independently.
MPT_LOG = open(WORK / 'moneyprinterturbo_streamlit.log', 'w', encoding='utf-8')
mpt_env = os.environ.copy()
mpt_env['INDICF5_API_URL'] = 'http://127.0.0.1:8765'
mpt_env['INDICF5_REFERENCE_DIR'] = str(Path(INDICF5_REFERENCE_AUDIO).resolve().parent)
MPT_PROCESS = subprocess.Popen([sys.executable, str(MPT_ROOT / 'kaggle' / 'mpt_kaggle.py')], cwd=str(MPT_ROOT), env=mpt_env, stdout=MPT_LOG, stderr=subprocess.STDOUT)
time.sleep(8)
if MPT_PROCESS.poll() is not None:
    MPT_LOG.flush()
    print((WORK / 'moneyprinterturbo_streamlit.log').read_text(encoding='utf-8', errors='replace')[-8000:])
    raise RuntimeError('MoneyPrinterTurbo UI process exited during startup.')
print('MoneyPrinterTurbo UI process started on port 8501. Process ID:', MPT_PROCESS.pid)
print('IndicF5 process ID:', INDICF5_PROCESS.pid)

def generate_from_json(json_path, cleanup=False, output_dir=None):
    """Run Manual Timeline JSON rendering directly without opening Streamlit."""
    json_path = Path(json_path).expanduser().resolve()
    if not json_path.is_file(): raise FileNotFoundError(json_path)
    if not requests.get('http://127.0.0.1:8765/health', timeout=5).json().get('ok'):
        raise RuntimeError('IndicF5 API is not healthy; refusing to start full render.')
    cmd = [sys.executable, str(MPT_ROOT / 'kaggle' / 'generate_from_json.py'), str(json_path)]
    if cleanup: cmd.append('--cleanup')
    if output_dir: cmd.extend(['--output-dir', str(Path(output_dir).resolve())])
    env = os.environ.copy()
    env['INDICF5_API_URL'] = 'http://127.0.0.1:8765'
    env['INDICF5_REFERENCE_AUDIO'] = INDICF5_REFERENCE_AUDIO
    env['INDICF5_REFERENCE_TEXT'] = INDICF5_REFERENCE_TEXT
    result = subprocess.run(cmd, cwd=str(MPT_ROOT), env=env, text=True)
    if result.returncode != 0: raise RuntimeError(f'JSON video generation failed with exit code {result.returncode}')

print('Ready. Example: generate_from_json("/kaggle/working/my_video.json", cleanup=False)')


In [ ]:
# Optional: expose the Streamlit UI through a Cloudflare Quick Tunnel.
# WARNING: anyone with the generated public URL may be able to reach the UI. Leave disabled unless needed.
if ENABLE_PUBLIC_TUNNEL:
    cloudflared = WORK / 'cloudflared'
    subprocess.run(['bash', '-lc', f'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {cloudflared} && chmod +x {cloudflared}'], check=True)
    TUNNEL_LOG = open(WORK / 'cloudflared.log', 'w', encoding='utf-8')
    TUNNEL_PROCESS = subprocess.Popen([str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:8501', '--no-autoupdate'], stdout=TUNNEL_LOG, stderr=subprocess.STDOUT)
    import re
    public_url = None
    for _ in range(60):
        time.sleep(2); TUNNEL_LOG.flush()
        logs = (WORK / 'cloudflared.log').read_text(encoding='utf-8', errors='replace')
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', logs)
        if match: public_url = match.group(0); break
    print('Streamlit public URL:', public_url or 'Tunnel URL not found; inspect cloudflared.log')
else:
    print('Public tunnel disabled. Set ENABLE_PUBLIC_TUNNEL=True in the configuration cell and rerun this cell if you need a browser-accessible UI link.')


## Generate a video from JSON

Upload your Manual Timeline JSON into `/kaggle/working` (or attach a dataset containing it), then run a code cell such as:

```python
generate_from_json('/kaggle/working/my_video.json', cleanup=False)
```

Use `cleanup=True` only after you are happy with the result. Final videos are saved under `storage/manual_timeline/<project_id>/` by default. Copy final MP4 files to `/kaggle/working/outputs` if you want them easy to download. Kaggle sessions are temporary, so download outputs or save them to persistent storage.
